# Recipe Generator - Llama 3.2 1B Fine-tuning with LoRA (QLoRA)
Run in Google Colab with GPU: `Runtime > Change runtime type > T4 GPU`

## Step 0: Install libraries

In [1]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.9 MB/s eta 0:00:00


In [2]:
import torch
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

## Step 1: Model + dataset names

In [3]:
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
DATASET_NAME = "tengomucho/all-recipes-split"

## Step 3: Load dataset

In [4]:
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(raw_dataset)
print(raw_dataset[0])

README.md:   0%|          | 0.00/365 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  157MB            

data/train-00001-of-00004.parquet: downloading bytes:           |  0.00B            

...

data/train-00002-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2147248 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'ingredients', 'directions'],
    num_rows: 2147248
})
{'title': 'No-Bake Nut Cookies', 'ingredients': '- 1 c. firmly packed brown sugar\n- 1/2 c. evaporated milk\n- 1/2 tsp. vanilla\n- 1/2 c. broken nuts (pecans)\n- 2 Tbsp. butter or margarine\n- 3 1/2 c. bite size shredded rice biscuits', 'directions': '- In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.\n- Stir over medium heat until mixture bubbles all over top.\n- Boil and stir 5 minutes more. Take off heat.\n- Stir in vanilla and cereal; mix well.\n- Using 2 teaspoons, drop and shape into 30 clusters on wax paper.\n- Let stand until firm, about 30 minutes.'}


In [5]:
NUM_EXAMPLES = 800  # kept small for a fast run - raise later for a stronger model
raw_dataset = raw_dataset.shuffle(seed=42).select(range(min(NUM_EXAMPLES, len(raw_dataset))))

## Step 4: Clean and format the data

In [6]:
def to_text(field):
    if isinstance(field, list):
        return "\n".join(str(x).strip() for x in field if str(x).strip())
    return str(field).strip()

def is_valid(example):
    return bool(example.get("title")) and bool(example.get("ingredients")) and bool(example.get("directions"))

def format_example(example):
    ingredients_text = to_text(example["ingredients"])
    directions_text = to_text(example["directions"])
    title = str(example["title"]).strip()

    prompt = (
        "Below is a list of ingredients. Write a recipe using them.\n\n"
        f"Ingredients:\n{ingredients_text}\n\nRecipe:"
    )
    completion = (
        f" Title: {title}\n"
        f"Ingredients:\n{ingredients_text}\n"
        f"Directions:\n{directions_text}"
    )
    return {"prompt": prompt, "completion": completion}

clean_dataset = raw_dataset.filter(is_valid)
clean_dataset = clean_dataset.map(format_example, remove_columns=clean_dataset.column_names)

Filter:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

## Step 5: Train / validation / test split (80/10/10)

In [7]:
split_1 = clean_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_1["train"]

split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)
val_dataset = split_2["train"]
test_dataset = split_2["test"]

## Step 6: Load model + tokenizer with 4-bit quantization + LoRA

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


## Step 7: Tokenize

In [9]:
MAX_LENGTH = 256  # shortened for speed

def tokenize_function(example):
    full_text = example["prompt"] + example["completion"] + tokenizer.eos_token
    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_tokenized = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, remove_columns=val_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_function, remove_columns=test_dataset.column_names)

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

## Step 8: Training configuration (fast settings)

In [10]:
training_args = TrainingArguments(
    output_dir="./llama-recipe-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,
    report_to="none",
    load_best_model_at_end=True,
)

## Step 9: Fine-tune (this is the slow step)

In [11]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.164981,1.062536


TrainOutput(global_step=40, training_loss=1.19855797290802, metrics={'train_runtime': 106.855, 'train_samples_per_second': 5.989, 'train_steps_per_second': 0.374, 'total_flos': 959991628431360.0, 'train_loss': 1.19855797290802, 'epoch': 1.0})

## Step 10: Evaluate

In [12]:
eval_results = trainer.evaluate(eval_dataset=test_tokenized)
print(eval_results)
print(f"Test perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
1.164981,1.085321,1


{'eval_loss': 1.085321068763733}
Test perplexity: 2.96


## Step 11: Save the trained model

In [13]:
SAVE_DIR = "recipe_llama_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

('recipe_llama_lora/tokenizer_config.json',
 'recipe_llama_lora/chat_template.jinja',
 'recipe_llama_lora/tokenizer.json')

## Step 12: Inference function

In [14]:
def generate_recipe(ingredients_list, max_new_tokens=250):
    ingredients_text = "\n".join(ingredients_list)
    prompt = (
        "Below is a list of ingredients. Write a recipe using them.\n\n"
        f"Ingredients:\n{ingredients_text}\n\nRecipe:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Test with your own ingredients

In [15]:
def apna_test():
    user_input = input("rice","chicken","garlic")
    ingredients_list = [x.strip() for x in user_input.split(",") if x.strip()]
    print("\nRecipe ban rahi hai...\n")
    print(generate_recipe(ingredients_list, max_new_tokens=300))

apna_test()

Apne ingredients comma se alag likho: rice, garlic


[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Recipe ban rahi hai...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Below is a list of ingredients. Write a recipe using them.

Ingredients:
rice
garlic

Recipe: Title: Garlic Rice
Ingredients:
rice
garlic
toasted almonds
salt
pepper
cumin
fresh parsley
toasted almonds
salt
pepper
cumin
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
canned tomato paste
olive oil
garlic
fresh parsley
c

## Download the trained model as a zip

In [16]:
from google.colab import files
import shutil
shutil.make_archive("recipe_llama_lora", 'zip', "recipe_llama_lora")
files.download("recipe_llama_lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>